# 03 — Git pour développeurs SVN

Ce notebook couvre les différences fondamentales entre SVN et Git.  
Objectif : comprendre Git en profondeur, pas juste mémoriser des commandes.

**Prérequis :** avoir complété `02-conda.ipynb`

## 1. Architecture fondamentale : centralisé vs distribué

### SVN — Architecture centralisée

```
        [SERVEUR SVN]
        ┌───────────┐
        │ Dépôt     │  ← L'unique source de vérité
        │ complet   │
        │ r1, r2... │
        └─────┬─────┘
              │ réseau (HTTP, SVN://)
    ┌─────────┼─────────┐
    │         │         │
┌───┴───┐ ┌───┴───┐ ┌───┴───┐
│Alice  │ │Bob    │ │Carol  │
│working│ │working│ │working│
│copy   │ │copy   │ │copy   │
│(partiel)│ │(partiel)│ │(partiel)│
└───────┘ └───────┘ └───────┘
```

Chaque développeur a une **working copy partielle** — juste les fichiers du répertoire courant.  
L'historique complet vit uniquement sur le serveur.  
Si le serveur tombe : tu peux lire tes fichiers locaux, mais tu ne peux pas consulter l'historique, créer une branche, ni commiter.

---

### Git — Architecture distribuée

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ Alice        │     │ GitHub       │     │ Bob          │
│ ┌──────────┐ │     │ ┌──────────┐ │     │ ┌──────────┐ │
│ │ Repo     │ │     │ │ Repo     │ │     │ │ Repo     │ │
│ │ COMPLET  │◄├─────┤►│ COMPLET  │◄├─────┤►│ COMPLET  │ │
│ │ + history│ │push/│ │ + history│ │push/│ │ + history│ │
│ └──────────┘ │pull │ └──────────┘ │pull │ └──────────┘ │
└──────────────┘     └──────────────┘     └──────────────┘
```

Chaque développeur possède un **clone complet** — tout l'historique depuis le début du projet.  
GitHub n'est qu'un repo parmi d'autres, désigné comme "remote" par convention.  
Tu peux travailler offline : consulter l'historique, créer des branches, commiter — tout sans réseau.

| Opération | SVN | Git |
|-----------|-----|-----|
| Voir l'historique | `svn log` → requête réseau | `git log` → lecture locale |
| Créer une branche | opération réseau | opération locale |
| Commiter | envoi immédiat au serveur | stockage local |

## 2. Modèle de données : deltas vs snapshots

### SVN — Stockage par delta (différences successives)

SVN stocke **ce qui a changé** entre chaque révision. Pour reconstruire l'état à r5, SVN rejoue la chaîne depuis le début — le delta *est* le modèle de données.

```
r1: [contenu complet de hello.py]
r2: [delta: ligne 3 modifiée]
r3: [delta: lignes 7-9 ajoutées]
r4: [delta: ligne 3 supprimée]
r5: [delta: ligne 1 modifiée]

Pour lire hello.py à r5 : r1 + delta(r2) + delta(r3) + delta(r4) + delta(r5)
```

---

### Git — Stockage par snapshot complet

Git stocke le **fichier complet** à chaque fois qu'il change. Si un fichier n'a pas changé entre deux commits, Git stocke un pointeur vers le snapshot existant — pas de duplication.

```
Commit A :  [hello.py complet v1] [utils.py complet v1]
                │                        │
Commit B :  [pointeur → v1]       [utils.py complet v2]  ← utils.py a changé
                │                                 │
Commit C :  [hello.py complet v2] [pointeur → v2]         ← hello.py a changé
```

**Analogie C++ :** SVN, c'est reconstruire une `struct` en appliquant une série de patches.  
Git, c'est lire la `struct` directement en mémoire — l'accès est immédiat, peu importe la profondeur de l'historique.

---

### Est-ce que Git prend plus d'espace disque ?

En théorie, un snapshot complet prend plus de place qu'un delta. Mais Git compense avec les **packfiles** : une compression interne automatique qui rapproche les snapshots de la taille de deltas. Cette compression est plus agressive que celle de SVN car Git peut comparer des fichiers *différents* entre eux, pas seulement les révisions du même fichier.

En pratique, les deux systèmes sont dans le même ordre de grandeur. L'exception : les **fichiers binaires** (images, modèles ML) — un binaire qui change souvent génère un snapshot complet inutilisable par la compression delta. C'est pour ça qu'existe Git LFS, mais c'est un sujet séparé.

---

### Complexité d'accès : notation Big O

Le tableau ci-dessous utilise la notation **Big O**, standard en informatique pour exprimer la quantité de travail selon la taille des données.

- **O(n)** : le travail croît proportionnellement à `n`. Ici, `n` = le nombre de révisions. Plus le repo est vieux, plus c'est lent.
- **O(1)** : le travail est constant, peu importe la taille. En C, c'est l'équivalent d'accéder à `tableau[i]` — calcul d'adresse direct, toujours le même coût.

| | SVN | Git |
|--|-----|-----|
| Modèle de données | Delta (rejoué à chaque lecture) | Snapshot complet |
| Accès à un état ancien | O(n) — rejoue tous les deltas | O(1) — lit le snapshot directement |
| Compression interne | Delta par fichier | Packfiles (delta cross-fichiers) |
| Espace disque | Comparable en pratique | Comparable en pratique |

# 3. Les branches : copie physique vs pointeur léger

---

## SVN — Une branche est une copie de répertoire

En SVN, une branche est littéralement une copie du répertoire dans le dépôt :

```
Dépôt SVN :
  /trunk/           ← développement principal
  /branches/
    /feature-auth/  ← copie de /trunk au moment du branch
    /bugfix-42/     ← copie de /trunk au moment du branch
  /tags/
    /v1.0/          ← copie figée (convention, pas protégée)
```

```bash
# SVN : créer une branche = copie réseau côté serveur
svn copy https://svn.example.com/trunk \
         https://svn.example.com/branches/feature-auth \
         -m "Création branche feature-auth"
# → opération réseau, crée une révision, copie les fichiers côté serveur
```

SVN utilise une **lazy copy** (copie sur écriture côté serveur) : à la création de la branche, aucun fichier n'est copié physiquement. SVN crée des **références vers les nodes existants** du dépôt — exactement comme `std::string` avec COW en C++. La copie réelle n'arrive que lors du premier commit sur un fichier donné.

**Au moment de la copie :** SVN ne copie *rien* physiquement. Il crée juste un **pointeur** (une référence) vers les fichiers originaux dans le dépôt. C'est instantané et ne prend presque pas de place.

```
trunk/                →  [fichier A, révision 42]
                          [fichier B, révision 42]

branches/ma-branche/  →  pointe vers les mêmes blocs
```

**Au moment d'une modification :** C'est seulement quand tu **modifies** un fichier dans la branche que SVN crée une vraie copie de ce fichier, uniquement pour celui-là.

```
Dépôt SVN après svn copy :

/trunk/                       → node_id: 42 (contenu réel)
/branches/feature-auth/       → node_id: 42 (même pointeur — rien n'est copié)

Premier commit dans feature-auth :
/branches/feature-auth/hello.py  → node_id: 99 (nouveau node, contenu réel)
/branches/feature-auth/utils.py  → node_id: 42 (toujours le même pointeur)
```

La branche ne coûte presque rien à la création. Le coût en espace disque arrive **au fil des commits**, fichier par fichier.

**Pourquoi "lazy" (paresseux) ?** Le terme vient du fait que SVN **reporte le travail** au dernier moment possible — il ne fait la vraie copie que si c'est absolument nécessaire (à l'écriture). C'est un principe de programmation classique : *"ne fais pas maintenant ce que tu peux éviter de faire"*.

**Les bénéfices concrets :**

| Avantage | Explication |
|---|---|
| 🚀 **Rapidité** | Créer une branche est quasi-instantané, peu importe la taille du projet |
| 💾 **Espace disque** | Le dépôt ne grossit que proportionnellement aux *vraies* modifications |
| 🏷️ **Tags bon marché** | Un tag (snapshot) ne coûte presque rien à créer |

---

## Le problème de merge avant SVN 1.5 — et ce qu'il révèle

Pour fusionner deux branches, il faut savoir exactement quels deltas appliquer — ni trop, ni trop peu. Avant SVN 1.5, SVN ne stockait **nulle part** l'information "j'ai déjà mergé jusqu'à la révision r42". Cette métadonnée n'existait tout simplement pas dans le modèle de données.

```
trunk:   r1 ── r2 ── r3 ── r4 ── r5 ── r6
                      │                  ↑
                      │            tu veux merger ici
                   branch:
                      └── r3 ── r4 ── r5

Question : quels deltas appliquer sur trunk ?
→ r4 et r5 de la branche uniquement (r3 est le point de divergence)
→ SVN < 1.5 ne savait pas ça automatiquement — c'était à TOI de t'en souvenir
```

En pratique, tu devais noter manuellement les numéros de révision déjà mergés et spécifier la plage à la main :

```bash
# SVN < 1.5 : merger manuellement en précisant la plage exacte
svn merge -r 3:5 https://svn.example.com/branches/feature-auth
# Si tu oubliais ou te trompais → conflits ou double-application de deltas
```

Ce n'est pas un simple bug ou un oubli de développeur — c'est la **conséquence directe du modèle de données de SVN**. Puisqu'une branche est juste une copie de répertoire dans un dépôt à révisions globales linéaires, SVN n'a aucun moyen *structurel* de savoir d'où vient quoi. Il faut compenser par une métadonnée externe.

SVN 1.5 a introduit les **mergeinfo** — une propriété `svn:mergeinfo` stockée dans les fichiers qui enregistre automatiquement les révisions déjà mergées. Ça a résolu le problème en pratique, mais c'est une métadonnée de compensation ajoutée après coup, fragile à manipuler.

> **La leçon sous-jacente :** un mauvais modèle de données force à ajouter des métadonnées externes pour compenser ses lacunes. Un bon modèle de données rend ces métadonnées inutiles.

---

## Git — Une branche est un pointeur de 41 octets

En Git, une branche n'est qu'un **fichier texte contenant le hash SHA-1** du commit pointé :

```
.git/refs/heads/
  main          ← fichier contenant "a3f8c2d1..."
  feature-auth  ← fichier contenant "7b2e9f4c..."
  bugfix-42     ← fichier contenant "c1d4a8e2..."
```

```bash
# Git : créer une branche = écrire 41 octets sur disque
git branch feature-auth
# ou, créer et basculer dessus immédiatement :
git checkout -b feature-auth
# → opération locale, instantanée, zéro réseau
```

```
Historique des commits :

a3f8c2d ← main (pointe ici)
    │
7b2e9f4 ← feature-auth (pointe ici)
    │
c1d4a8e
    │
b9f3d1c
```

**Switching de branche :** Git met à jour les fichiers du working directory pour correspondre au snapshot du commit pointé. Il ne "copie" rien — il restaure un snapshot.

**Conséquence sur les merges :** Git n'a jamais eu le problème de SVN. Chaque commit connaît son parent via le hash SHA-1 — l'ancêtre commun de deux branches se calcule nativement en remontant le graphe de commits. C'est **structurel**, pas une métadonnée ajoutée. Ce n'est pas que Git a "mieux codé" la même chose : c'est que Git a **pensé le merge dès la conception du modèle de données**.

---

## Comparaison finale

| | SVN | Git |
|---|---|---|
| **Une branche, c'est quoi ?** | Une copie de répertoire | Un pointeur de 41 octets |
| **Coût de création** | Quasi-nul (lazy copy) | Quasi-nul (41 octets) |
| **Opération** | Réseau + révision serveur | Locale, zéro réseau |
| **Merge tracking** | Métadonnée externe (`svn:mergeinfo`) ajoutée en SVN 1.5 | Structurel via le graphe de commits SHA-1 |
| **Ancêtre commun** | Calculé à partir d'une métadonnée fragile | Calculé nativement en remontant le DAG |

La création d'une branche coûte presque rien dans les deux cas — ce n'est pas là que ça diverge vraiment. La vraie ligne de fracture, c'est le merge tracking :

- **SVN** : modèle linéaire (révisions globales) → merge tracking **impossible nativement** → compensation par métadonnée externe
- **Git** : modèle graphe (commits avec parents) → merge tracking **gratuit et structurel**

## 4. Workflow de commit : SVN en 1 étape vs Git en 2 étapes

### SVN — Commit direct au serveur

```
Working copy  ──── svn commit ────►  Serveur SVN
(fichiers modifiés)                  (nouvelle révision r42, visible par tous)
```

```bash
# SVN : modifier puis commiter = publier immédiatement
nano hello.py
svn commit -m "Fix bug in hello.py"
# → requête réseau, génère r42 sur le serveur, visible par tous immédiatement
```

Sans accès réseau → impossible de commiter.  
Commit trop tôt → tout le monde voit ton travail inachevé.

---

### Git — Commit local, puis push séparé

```
Working       ──── git add ────►  Staging  ──── git commit ────►  Repo local  ──── git push ────►  Remote
directory                          Area                            (.git/)                          (GitHub)
(fichiers modifiés)               (index)                         (nouveau commit)
```

```bash
# Git : modifier, stager, commiter localement, puis pousser quand prêt
nano hello.py
nano utils.py

git add hello.py              # staging area : seulement hello.py
git commit -m "Fix bug"       # commit LOCAL — personne ne voit encore
# ... continuer à travailler, faire d'autres commits ...
git push origin main          # maintenant GitHub est mis à jour
```

La séparation commit/push est puissante : tu peux faire 10 commits locaux pour affiner  
ton travail, puis les envoyer en une fois quand ils sont prêts.

## 5. La Staging Area — concept absent en SVN (mais souvent mal expliqué)

La Staging Area (aussi appelée **Index**) est une zone intermédiaire entre le working directory et le repo Git.  
Elle est souvent présentée comme une simple « sélection de fichiers avant le commit » — ce qui est trompeur, car SVN le permet aussi.  
La vraie différence est plus profonde.

---
### Ce que SVN permet déjà

SVN permet tout à fait de sélectionner des fichiers spécifiques au moment du commit :

In [ ]:
# SVN — sélection de fichiers spécifiques au commit
svn commit auth.py utils.py -m "Fix authentication bug #42"
svn commit ui.py templates/ -m "Add user profile page"

> La sélection de fichiers n'est donc **pas** l'apport de Git. Ce qui change, c'est la **nature** de cette sélection.

---
### Ce que Git ajoute vraiment

#### 1. Un état persistant et inspectable, pas un argument de commande

En SVN, la liste de fichiers passée au `commit` est **éphémère** — c'est un paramètre de ligne de commande, et rien d'autre.  
Il n'existe pas de « commit en cours de construction » que tu peux assembler progressivement, suspendre, inspecter, et corriger.

En Git, l'index est un **snapshot intermédiaire vivant** :

In [ ]:
git add auth.py               # l'index contient auth.py
# ... tu continues à travailler, tu relances tes tests ...
git add utils.py              # tu enrichis l'index
git diff --staged             # tu inspectes exactement ce qui va partir
git restore --staged auth.py  # tu retires auth.py si tu changes d'avis
git commit -m "Fix authentication bug #42"

> Tu construis ton commit comme un **brouillon**, que tu affines avant de le finaliser.

---
#### 2. La granularité intra-fichier — là où SVN ne peut pas suivre

C'est la **vraie différence structurelle**. Git peut stager des **blocs de code (hunks) à l'intérieur d'un même fichier** avec `git add -p` :

In [ ]:
# Tu as modifié 3 sections dans utils.py :
#   section A → bug fix
#   section B → refactoring non lié
#   section C → bug fix aussi

git add -p utils.py
# Git affiche chaque bloc un par un et te demande quoi faire :
#   Hunk A → y  (stager)
#   Hunk B → n  (ignorer pour l'instant)
#   Hunk C → y  (stager)

git commit -m "Fix authentication bug #42"
# ✅ Le commit contient les blocs A et C de utils.py
# ⏳ Le bloc B reste dans ton working directory, non commité

> **SVN ne peut pas faire ça.** Sa granularité maximale est le fichier entier.

---
#### 3. L'index capture un contenu précis, pas un pointeur vers un fichier

Ce point est souvent contre-intuitif : `git add` ne note pas *« ce fichier sera commité »*, il **photographie le contenu du fichier à cet instant précis**.

In [ ]:
echo "version 1" > hello.py
git add hello.py              # l'index contient "version 1"

echo "version 2" > hello.py  # tu modifies le fichier APRÈS le git add

git status
# → hello.py apparaît à la fois dans :
#     "Changes to be committed"       ← l'index a "version 1"
#     "Changes not staged for commit" ← le disque a "version 2"

# Le commit contiendra "version 1", pas "version 2"

> L'index a capturé le **contenu au moment du `add`**, pas l'état actuel du fichier sur le disque.

---
### Vue d'ensemble

```
┌─────────────────┐              ┌──────────────────────┐              ┌─────────────────┐
│ Working Dir     │              │ Staging Area (Index) │              │ Repository      │
│                 │              │                      │              │ (.git/)         │
│ hello.py  (M)   │──git add────►│ hello.py  ✓          │──git commit─►│                 │
│ utils.py  (M)   │──git add -p─►│ utils.py (hunks A+C) │              │ snapshot        │
│ config.py (M)   │  (pas stagé) │                      │              │ précis          │
└─────────────────┘              └──────────────────────┘              └─────────────────┘
                                           ↑
                                 Tu construis ici ton commit
                                 progressivement, avant de le figer
```

---
### États des fichiers en Git

```
Untracked  ──── git add ────►  Staged  ──── git commit ────►  Committed
(nouveau fichier,              (contenu capturé dans            (snapshot dans
 Git ne le connaît pas)         l'index, prêt à partir)          le repo .git/)

Modified   ──── git add ────►  Staged
(fichier connu, contenu changé)
```

---
### Résumé — Différences SVN → Git

| Capacité | SVN | Git |
|:---|:---:|:---:|
| Sélectionner des fichiers spécifiques au commit | ✅ | ✅ |
| Construire le commit progressivement avant de le finaliser | ❌ | ✅ |
| Inspecter et modifier la sélection avant le commit | ❌ | ✅ |
| Stager des blocs précis à l'intérieur d'un fichier | ❌ | ✅ `git add -p` |
| Capturer un contenu précis, pas juste un pointeur de fichier | ❌ | ✅ |

---
### Commandes utiles

In [ ]:
git status                     # état de chaque fichier
git diff                       # working directory vs staging area
git diff --staged              # staging area vs dernier commit
git add -p <fichier>           # staging interactif hunk par hunk
git restore --staged <fichier> # retirer un fichier de l'index

---
> **💡 L'essentiel :** La staging area n'est pas là pour sélectionner des fichiers — SVN fait ça.  
> Elle est là pour composer un commit **chirurgicalement**, à la granularité du bloc de code,  
> et te laisser le construire **progressivement** avant de le graver dans l'historique.

# 6. Commit local vs serveur — Identifiants et visibilité

## SVN — Le numéro de révision est global et séquentiel

En SVN, chaque révision a un numéro entier global et séquentiel assigné par le serveur : r1, r2, r3... Il est impossible d'avoir un commit "local" — commiter = publier immédiatement.

```
r1  ── r2  ── r3  ── r4  ── r5
                              ↑
                         dernier commit
                         (visible par tous dès que créé)
```

---

## Git — Le hash SHA-1 est universel et calculé localement

En Git, chaque commit est identifié par un hash SHA-1 de **160 bits**, calculé à partir de :
- le contenu des fichiers snapshottés
- le nom et email de l'auteur
- la date et heure du commit
- le hash du commit parent

Ce hash est identique sur toutes les machines qui ont ce commit — sans coordination avec un serveur.

```bash
git log --oneline
a3f8c2d Fix bug in auth module        ← commit local, pas encore pushé
7b2e9f4 Add user profile feature      ← commit local, pas encore pushé
c1d4a8e Initial project structure     ← commit déjà sur GitHub
```

---

## Mais comment le hash peut-il rester unique sans serveur central ?

C'est une question légitime : si deux développeurs génèrent des hashes indépendamment, ne risquent-ils pas d'obtenir le même par hasard ?

La réponse est dans les chiffres. SHA-1 produit **2¹⁶⁰ valeurs possibles**, soit environ :

```
1 461 501 637 330 902 918 203 684 832 716 283 019 655 932 542 976
```

Pour avoir une probabilité de collision de 50 % entre commits générés aléatoirement *(paradoxe des anniversaires)*, il faudrait environ **2⁸⁰ ≈ 1,2 × 10²⁴ commits** — soit un milliard de milliards de fois plus que tous les commits créés depuis l'invention de Git.

De plus, deux commits avec un contenu différent (même d'un seul caractère, ou d'une milliseconde d'écart dans l'horodatage) produisent des hashes **radicalement différents**. Il ne s'agit pas d'un tirage aléatoire indépendant : le hash *encode* la réalité du commit.

> En pratique, une collision SHA-1 accidentelle sur des commits Git est considérée comme **impossible** dans un horizon humain. C'est pour cela que Git peut garantir l'unicité globale sans serveur.

*(Note : Git migre progressivement vers SHA-256 pour 2²⁵⁶ valeurs possibles, non pas à cause de collisions accidentelles, mais pour se prémunir contre d'éventuelles attaques cryptographiques délibérées.)*

---

## Conséquence pratique

Deux développeurs peuvent travailler offline une semaine, créer des dizaines de commits chacun, puis synchroniser — Git peut fusionner leurs historiques car les hashes garantissent l'unicité globale de chaque commit sans serveur central.

| Aspect | SVN | Git |
|---|---|---|
| Identifiant de commit | `r42` (entier séquentiel serveur) | `a3f8c2d` (SHA-1 universel) |
| Offline commit | Impossible | Natif |
| Offline log | Impossible | Natif |
| Visibilité après commit | Immédiate pour tous | Locale uniquement jusqu'au push |
| Garantie d'unicité | Serveur central | Mathématique (2¹⁶⁰ valeurs) |

## 7. Tableau récapitulatif

| Concept | SVN | Git |
|---------|-----|-----|
| **Architecture** | Centralisée — un seul serveur | Distribuée — chaque clone est complet |
| **Historique local** | Non — sur le serveur uniquement | Oui — complet sur chaque machine |
| **Modèle de stockage** | Deltas (différences successives) | Snapshots (état complet du projet) |
| **Branches** | Copie de répertoire côté serveur | Pointeur de 41 octets, local |
| **Création de branche** | Opération réseau, côté serveur | Instantanée, locale |
| **Identifiant de commit** | Entier séquentiel global (`r42`) | Hash SHA-1 universel (`a3f8c2d`) |
| **Commit** | = publication immédiate au serveur | = sauvegarde locale uniquement |
| **Publication** | Implicite dans `svn commit` | Explicite via `git push` |
| **Staging Area** | Inexistante | Obligatoire (index) |
| **Travail offline** | Lecture seule | Complet (commit, branch, log...) |
| **Merge** | Difficile (historique mal tracké avant SVN 1.5) | Fiable (ancêtre commun connu) |
| **Checkout d'un commit ancien** | Lent — rejoue tous les deltas | Rapide — lecture directe du snapshot |

## 8. Métaphore finale

**SVN, c'est Google Docs :**  
Le document vit sur le serveur. Tu ouvres une connexion, tu modifies, tu sauvegardes —  
et tout le monde voit la sauvegarde immédiatement. Sans connexion : lecture seule.

**Git, c'est un carnet de notes physique avec photocopieuse :**  
Chaque développeur a son carnet complet. Tu écris dedans (commit local) sans connexion.  
Quand tu veux partager, tu photocopies les nouvelles pages et tu les envoies (push).  
Les autres font pareil et vous fusionnez vos carnets (merge/pull).  
Le carnet partagé (GitHub) n'est qu'un carnet désigné comme référence par convention —  
il n'a aucun statut spécial techniquement.

# Cohérence du code sur Git à travers plusieurs développeurs

## Les deux types de conflits

Il faut distinguer deux problèmes très différents :

**1. Le conflit de texte** — deux personnes ont modifié la même ligne
→ Git le détecte automatiquement et bloque le merge jusqu'à résolution.

**2. Le conflit de philosophie** — deux personnes ont fait évoluer le code dans des directions incompatibles sans toucher les mêmes lignes
→ Git ne le voit pas du tout. C'est le vrai danger.

**Exemple du deuxième cas :** un dev refactorise toute la gestion des erreurs vers un système de `Result<T>`, pendant qu'un autre ajoute 10 nouvelles fonctions avec des `try/catch` classiques. Le merge passe sans conflit Git... mais le code résultant est incohérent.

---

## Ce que les équipes mettent en place

### Branches et intégration courte

La règle d'or est de **ne pas laisser les branches diverger longtemps**. Plus une branche vit longtemps, plus la réconciliation est douloureuse. Les équipes modernes visent des branches qui durent **heures ou jours**, pas semaines.

```
main ──────────────────────────────────►
  └── feature/auth (3 jours max) ──┘
  └── feature/api  (2 jours max) ────┘
```

### Pull Requests et code review

Avant qu'une branche soit fusionnée dans `main`, un ou plusieurs développeurs **relisent le code**. C'est ici que les conflits de philosophie sont détectés — pas par Git, mais par des humains.

> La Pull Request est le vrai filet de sécurité contre la divergence intellectuelle.

### Les règles de protection de branche

Sur GitHub/GitLab, `main` peut être configurée pour **bloquer tout merge direct** sans :
- au moins 1 approbation humaine
- les tests automatiques qui passent (CI/CD)
- la branche à jour avec `main`

### Tests automatiques (CI/CD)

Chaque push déclenche une suite de tests. Si deux philosophies entrent en collision, les tests cassent — même si Git n'a vu aucun conflit de texte. C'est le deuxième filet de sécurité.

### Conventions et architecture documentées

Les équipes sérieuses maintiennent des **Architecture Decision Records (ADR)** — des documents courts qui expliquent *pourquoi* une décision technique a été prise. Quand quelqu'un veut partir dans une direction différente, il lit les ADRs et comprend le contexte avant de diverger.

---

## La réalité du terrain

Pour être honnête : **les conflits de philosophie arrivent quand même**, surtout dans les grandes équipes. Voici ce qui les génère le plus souvent :

| Situation | Résultat typique |
|---|---|
| Équipe qui grandit vite | Nouveaux devs qui n'ont pas lu les conventions |
| Mauvaise communication | Deux features parallèles qui se marchent dessus |
| Pas de code review | Divergences non détectées pendant des semaines |
| Branches longues | Réconciliation douloureuse en fin de sprint |
| Pas de tests | Les incohérences passent en production |

---

## Ce que Git apporte vraiment

Git ne résout pas la cohérence intellectuelle — il donne juste les **outils pour l'organiser** :
- `git blame` pour comprendre pourquoi une ligne existe
- `git log` pour voir l'historique d'une décision
- les branches pour isoler les expérimentations
- les tags pour marquer les versions stables

La cohérence réelle vient du **processus humain** autour de Git : les reviews, les conventions, la communication, les tests. Git est l'infrastructure — pas le chef d'orchestre.

> En pratique, une équipe disciplinée avec de mauvais outils s'en sortira mieux qu'une équipe désorganisée avec les meilleurs outils du monde.

# Git Blame — Comprendre l'historique ligne par ligne

## Qu'est-ce que `git blame` ?

`git blame` répond à une question simple :

> *"Pour chaque ligne qui existe aujourd'hui dans ce fichier, quel commit l'a écrite en dernier ?"*

Ce n'est **pas** un outil pour accuser quelqu'un — c'est un outil de **compréhension du contexte**.

---

## Comment Git construit le résultat

Git stocke chaque commit comme un snapshot complet de tous les fichiers, avec ses métadonnées (auteur, date, message). `git blame` remonte l'historique et, pour chaque ligne, note le commit le plus récent qui l'a touchée.

### Exemple — Évolution d'un fichier sur 3 commits

**Commit `c4a1b8f` — Thomas — 1er août** (version initiale)
```python
1  def calculate_tax(order):
2      rate = 0.15
3      return order.total * rate
```

**Commit `7b2e9f4` — Sophie — 12 septembre** (remplace la ligne 2)
```python
1  def calculate_tax(order):
2      rate = TAX_RATES[order.region]   ← ligne modifiée
3      return order.total * rate
```

**Commit `e1c9d3a` — Marc — 3 novembre** (insère des lignes entre 2 et 3)
```python
1  def calculate_tax(order):
2      rate = TAX_RATES[order.region]
3      if date.hour == 23:              ← ligne ajoutée
4          rate = get_next_day_rate()   ← ligne ajoutée
5      return order.total * rate
```

Git remonte l'historique et attribue chaque ligne à son dernier commit :

```
Ligne 1 → jamais modifiée depuis Thomas  → c4a1b8f / Thomas / 1er août
Ligne 2 → modifiée en dernier par Sophie → 7b2e9f4 / Sophie / 12 septembre
Ligne 3 → ajoutée par Marc              → e1c9d3a / Marc   / 3 novembre
Ligne 4 → ajoutée par Marc              → e1c9d3a / Marc   / 3 novembre
Ligne 5 → jamais modifiée depuis Thomas → c4a1b8f / Thomas / 1er août
```

Ce que `git blame` affiche :

```
c4a1b8f (Thomas  2024-08-01)  def calculate_tax(order):
7b2e9f4 (Sophie  2024-09-12)      rate = TAX_RATES[order.region]
e1c9d3a (Marc    2024-11-03)      if date.hour == 23:
e1c9d3a (Marc    2024-11-03)          rate = get_next_day_rate()
c4a1b8f (Thomas  2024-08-01)      return order.total * rate
```

### D'où vient concrètement l'information ?

Tout vient de la base de données locale `.git/` dans ton projet :

```
ton-projet/
├── .git/                ← la base de données de tout l'historique
│   ├── objects/         ← tous les commits, fichiers, snapshots stockés
│   └── logs/            ← journal des mouvements de branches
├── tax_calculator.py
└── ...
```

`git blame` ne consulte pas GitHub — il lit directement `.git/objects/` en local, sans connexion internet.

---

## `git blame` vs `git log` — Quelle différence ?

Les deux puisent dans le même historique, mais le **point de vue est différent**.

### `git log` — Vue par commit (chronologique)

Répond à : *"Quels commits ont touché ce fichier, et quand ?"*

```bash
git log --oneline tax_calculator.py

e1c9d3a Marc    3 nov   Fix: apply next-day tax rates
7b2e9f4 Sophie  12 sep  Replace hardcoded rate with regional rates
c4a1b8f Thomas  1 août  Initial tax calculator
```

Tu vois l'histoire du fichier dans son ensemble — mais tu ne sais pas quelles lignes ont été touchées par qui. Tu dois ouvrir chaque commit un par un pour le savoir.

### `git blame` — Vue par ligne (état actuel)

Répond à : *"Pour chaque ligne qui existe aujourd'hui, quel commit l'a écrite ?"*

```bash
c4a1b8f Thomas   def calculate_tax(order):
7b2e9f4 Sophie       rate = TAX_RATES[order.region]
e1c9d3a Marc         if date.hour == 23:
e1c9d3a Marc             rate = get_next_day_rate()
c4a1b8f Thomas       return order.total * rate
```

Tu vois l'état actuel du fichier, annoté ligne par ligne. Tu sais immédiatement qui a écrit la ligne 3 sans ouvrir aucun commit.

### L'analogie concrète

Imagine un contrat Word modifié par 3 personnes :

- **`git log`** = la liste des versions du document : *"Version 1 Thomas, Version 2 Sophie, Version 3 Marc"* — tu dois ouvrir chaque version pour voir ce qu'ils ont changé.
- **`git blame`** = le mode **"Suivi des modifications"** de Word — tu vois le document tel qu'il est aujourd'hui, avec une couleur différente par auteur sur chaque paragraphe.

### Quand utiliser lequel ?

| Situation | Outil |
|---|---|
| *"Qu'est-ce qui a changé dans ce fichier au fil du temps ?"* | `git log` |
| *"Qui a écrit cette ligne précise qui bug ?"* | `git blame` |
| *"Pourquoi cette ligne existe ?"* | `git blame` → puis `git show <hash>` |

Les deux sont complémentaires : `git blame` te donne le **hash**, et tu utilises `git show` pour comprendre le **contexte** de ce hash.

---

## Exemple concret d'utilisation en équipe

### Le bug

Léa reçoit un rapport : *"Les commandes passées entre 23h et minuit ont un mauvais calcul de TVA."*

### Étape 1 — Localiser avec `git blame`

```bash
git blame tax_calculator.py
```

```
c4a1b8f (Thomas  2024-08-01)  def calculate_tax(order):
7b2e9f4 (Sophie  2024-09-12)      rate = TAX_RATES[order.region]
e1c9d3a (Marc    2024-11-03)      if date.hour == 23:             ← suspect
e1c9d3a (Marc    2024-11-03)          rate = get_next_day_rate()  ← suspect
c4a1b8f (Thomas  2024-08-01)      return order.total * rate
```

Léa identifie immédiatement que les lignes suspectes viennent du commit `e1c9d3a` de Marc.

### Étape 2 — Comprendre l'intention avec `git show`

```bash
git show e1c9d3a
```

```
Fix: apply next-day tax rates for late-night orders (ticket #312)

Some regions change their tax rate at midnight. Orders placed at 23h
were being charged the old rate. Applied next-day rate from 23h onward
as a workaround until we have a proper timezone solution.
```

Marc avait résolu un vrai problème, mais avec un workaround fragile — il ne gère pas les fuseaux horaires ni les minutes/secondes.

### Étape 3 — Réaction d'équipe

Léa ouvre une issue GitHub avec le hash, l'explication, et mentionne Marc et Sophie. Marc répond :

> *"J'avais laissé un TODO dans le ticket #312 pour ça — on n'avait pas le temps de gérer les timezones à ce moment."*

Sophie ajoute :

> *"Les `TAX_RATES` que j'avais mis en place supportent déjà les dates UTC, on peut s'appuyer dessus."*

En 20 minutes, l'équipe a reconstitué l'intention originale et la bonne direction — **sans réunion**.

### Étape 4 — Fix traçable

```bash
git commit -m "Fix tax rate calculation for orders near midnight

git blame on tax_calculator.py:47 revealed a workaround introduced
in e1c9d3a (ticket #312) that only handled hour==23 without timezone
or sub-minute precision.

Replaced with UTC-aware datetime comparison using existing TAX_RATES
date support (introduced by Sophie in 7b2e9f4).

Closes #387. See also #312."
```

Dans 6 mois, le prochain dev qui touchera cette fonction trouvera ce commit dans `git blame` et comprendra le contexte complet.

---

## Comparatif Git vs SVN

SVN possède l'équivalent de `git blame` : la commande `svn annotate` (ou `svn blame`). Le résultat est similaire en apparence, mais les différences sous le capot sont importantes.

### Comparaison des commandes

**Git :**
```bash
git blame tax_calculator.py

c4a1b8f (Thomas  2024-08-01)  def calculate_tax(order):
7b2e9f4 (Sophie  2024-09-12)      rate = TAX_RATES[order.region]
e1c9d3a (Marc    2024-11-03)      if date.hour == 23:
```

**SVN :**
```bash
svn annotate tax_calculator.py

     1  thomas   def calculate_tax(order):
     4  sophie       rate = TAX_RATES[order.region]
     7  marc         if date.hour == 23:
```

Le résultat est lisiblement identique — mais le numéro `7` est un **numéro de révision global du serveur**, pas un hash local.

### Différences fondamentales

| Aspect | Git (`git blame`) | SVN (`svn annotate`) |
|---|---|---|
| **Identifiant affiché** | Hash SHA-1 (`e1c9d3a`) | Numéro de révision serveur (`r7`) |
| **Connexion requise** | ❌ Non — tout est en local | ✅ Oui — doit interroger le serveur |
| **Vitesse** | Instantané | Dépend du réseau et du serveur |
| **Offline** | ✅ Fonctionne | ❌ Impossible |
| **Lien avec le message de commit** | `git show <hash>` en local | `svn log -r 7` via le serveur |
| **Portabilité du résultat** | Le hash est universel sur toutes les machines | Le numéro r7 n'a de sens que dans ce dépôt SVN |
| **Précision de l'historique** | Ligne par ligne avec arbre de commits complet | Ligne par ligne, mais historique centralisé |

### La différence clé : offline vs online

C'est la conséquence directe de l'architecture de chaque système :

- En **Git**, l'historique complet est cloné sur chaque machine. `git blame` peut donc tout calculer localement, sans réseau.
- En **SVN**, l'historique vit uniquement sur le serveur central. `svn annotate` doit le consulter pour chaque requête — si le serveur est down, la commande échoue.

```
Git :   [ta machine] ──── git blame ────► résultat  (pas de réseau)

SVN :   [ta machine] ──── svn annotate ──► [serveur SVN] ──► résultat
                                                ↑
                                     (réseau requis)
```

### La différence culturelle : hash vs numéro de révision

Quand Léa partage le hash `e1c9d3a` à Marc par message, Marc peut immédiatement faire `git show e1c9d3a` sur **sa propre machine** et voir le commit exact.

En SVN, si Léa dit "regarde la révision r7", Marc peut le faire aussi — mais seulement si le serveur est accessible, et le numéro `r7` est spécifique à ce dépôt SVN particulier. Sur un autre dépôt, r7 désigne quelque chose de complètement différent.

---

## Résumé final

| Question | Git | SVN |
|---|---|---|
| *"Qui a écrit cette ligne ?"* | `git blame` | `svn annotate` |
| Connexion requise | Non | Oui |
| Identifiant du commit | Hash universel (`e1c9d3a`) | Révision serveur (`r7`) |
| Comprendre le contexte | `git show <hash>` en local | `svn log -r 7` via serveur |
| Fonctionne offline | ✅ | ❌ |
| Philosophie | Historique distribué sur chaque machine | Historique centralisé sur le serveur |

> `git blame` est efficace dans une équipe où **blâmer n'est pas la culture**. Si les développeurs ont peur d'être montrés du doigt, ils écrivent des messages de commit vagues — et `git blame` perd toute sa valeur. L'outil est aussi bon que la discipline d'équipe qui l'entoure.